<a href="https://colab.research.google.com/github/Shreenitya/nitya/blob/main/Timelapse.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import geemap.geemap as map
import ee
ee.Authenticate()
ee.Initialize(project ='shreenitya0428')
m = map.Map()

In [2]:
Subarnarekha = ee.FeatureCollection('projects/shreenitya0428/assets/Subarnarekha')
m.addLayer(Subarnarekha, {}, 'Subarnarekha')


In [3]:
region = ee.Geometry.Polygon(
    [[[87.0844839, 21.8552299],
      [87.4500633, 21.8062620],
      [87.4617591, 21.4836805],
      [87.0486066, 21.4359486]
      ]],
    None,
    False)

In [4]:
NDWI = ee.ImageCollection("MODIS/MOD09GA_006_NDWI").select('NDWI')
m.addLayer(NDWI, {}, 'NDWI')
m

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGUI(childr…

In [5]:
NDWI_river = NDWI.map(lambda img: img.set('doy', ee.Date(img.get('system:time_start')).getRelative('day', 'year')));

In [17]:
distinctDOY = NDWI_river.filterDate('2018-04-01', '2019-04-01');

In [18]:
filter = ee.Filter.equals(leftField = 'doy', rightField = 'doy')
join = ee.Join.saveAll('doy_matches')
joincol = ee.ImageCollection(join.apply(distinctDOY, NDWI_river, filter))

In [19]:
comp = joincol.map(lambda img: ee.ImageCollection.fromImages(img.get('doy_matches')).reduce(ee.Reducer.mean()).set('doy', ee.Number(img.get('doy'))))

In [20]:
vis_params = {
  'min': 0.0,
  'max': 1.0,
  'palette': ['0000ff', '00ffff', 'ffff00', 'ff0000', 'ffffff'],
};

In [21]:
rgbvis = comp.map(lambda img: img.visualize(bands = ['NDWI_mean'], **vis_params).clip(Subarnarekha))

In [22]:
gifParams = {
  'region': region,
  'dimensions': 200,
  'crs': 'EPSG:4326',
  'framesPerSecond': 10,
  'format':'gif'
};

In [23]:
print(rgbvis.getVideoThumbURL(gifParams));

https://earthengine.googleapis.com/v1/projects/shreenitya0428/videoThumbnails/eecbf667a59a82697cbcbdca3461ebe3-f4288893a150a31ee322ebfdb8c4f880:getPixels
